# Domain 1 — Agents and Workflows (14.7%)

| Skill | Weight |
|---|---|
| Agent Construction with Claude | 5.3% |
| Agent Patterns and Frameworks | 4.9% |
| Agent Architecture | 4.5% |

## 1.1 Workflow or agent?

A **workflow** runs steps you decided in advance. A **agent** decides its
own steps at runtime. The decision rule:

- Steps known upfront, same every time → workflow. Cheaper, testable,
  debuggable, no runaway risk.
- Number and order of steps depends on what is discovered mid-task → agent.

The exam's trap is reaching for an agent when a workflow would do. Agents
cost more, fail in more ways, and are harder to test. Default to a workflow
and escalate only when the task genuinely needs runtime decisions.


In [ ]:
"""Shared setup. Export ANTHROPIC_API_KEY before launching Jupyter."""

import json
import os

import anthropic

client = anthropic.Anthropic()

# Current self-serve model IDs (verified against Anthropic docs, Sept 2026).
OPUS = "claude-opus-5"
SONNET = "claude-sonnet-5"
HAIKU = "claude-haiku-4-5-20251001"

MODEL = SONNET


def extract_text(response: anthropic.types.Message) -> str:
    """Concatenate text blocks, ignoring thinking and tool_use blocks."""
    return "".join(
        block.text for block in response.content if block.type == "text"
    )


print("API key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))


### A workflow: fixed, predetermined steps

Note what this buys you — each step is independently testable, the cost is
knowable in advance, and there is no loop to run away.


In [ ]:
def classify(claim_text: str) -> str:
    """Step 1: assign a category."""
    response = client.messages.create(
        model=HAIKU,
        max_tokens=10,
        system="Reply with one word: PROPERTY, AUTO, or LIABILITY.",
        messages=[{"role": "user", "content": claim_text}],
    )
    return extract_text(response).strip()


def extract_amount(claim_text: str) -> str:
    """Step 2: pull the monetary figure."""
    response = client.messages.create(
        model=HAIKU,
        max_tokens=20,
        system="Reply with only the dollar amount, or NONE.",
        messages=[{"role": "user", "content": claim_text}],
    )
    return extract_text(response).strip()


def summarise(claim_text: str) -> str:
    """Step 3: one-line summary."""
    response = client.messages.create(
        model=HAIKU,
        max_tokens=60,
        system="Summarise the claim in one short sentence.",
        messages=[{"role": "user", "content": claim_text}],
    )
    return extract_text(response).strip()


def claim_intake_workflow(claim_text: str) -> dict[str, str]:
    """Run a fixed three-step pipeline. No model decides the sequence."""
    return {
        "category": classify(claim_text),
        "amount": extract_amount(claim_text),
        "summary": summarise(claim_text),
    }


result = claim_intake_workflow(
    "Claim ABC-123: hail damaged the roof of the insured dwelling. "
    "Adjuster estimates $8,400 in repairs."
)
print(json.dumps(result, indent=2))


## 1.2 Agent Construction — the tool-use loop

An agent is a loop. Claude never executes a tool itself: it returns a
`tool_use` block, **your code** runs the function, and you send the output
back as a `tool_result` block in a new user turn. Repeat until
`stop_reason` is no longer `"tool_use"`.

Four things the exam checks in this loop:

1. Append the assistant's **entire** `response.content`, not just text.
2. Every `tool_use` block needs a matching `tool_result` with the same
   `tool_use_id`, in the very next user message.
3. Handle **multiple** `tool_use` blocks in one response (parallel calls).
4. Bound the loop. An unbounded agent loop is the classic production
   failure — and a classic wrong answer.


In [ ]:
CLAIMS_DB = {
    "ABC-123": {"status": "APPROVED", "amount": 8400, "peril": "hail"},
    "XYZ-789": {"status": "DENIED", "amount": 0, "peril": "flood"},
    "DEF-456": {"status": "PENDING", "amount": 15200, "peril": "fire"},
}

TOOLS = [
    {
        "name": "get_claim",
        "description": (
            "Retrieve status, payout amount, and peril for a single claim "
            "by its ID. Claim IDs have the format ABC-123 (three uppercase "
            "letters, hyphen, three digits). Call once per claim ID. Do "
            "not invent claim IDs that the user did not supply."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "claim_id": {
                    "type": "string",
                    "pattern": "^[A-Z]{3}-[0-9]{3}$",
                    "description": "The claim identifier, e.g. ABC-123.",
                }
            },
            "required": ["claim_id"],
        },
    }
]


def get_claim(claim_id: str) -> str:
    """Look up a claim. Returns an error string rather than raising."""
    record = CLAIMS_DB.get(claim_id)
    if record is None:
        return f"ERROR: no claim found with id {claim_id}"
    return json.dumps(record)


TOOL_REGISTRY = {"get_claim": get_claim}


In [ ]:
def run_agent(goal: str, max_turns: int = 6, verbose: bool = True) -> str:
    """Run a bounded tool-use loop and return the final text.

    max_turns is a hard stop. Without it, a model that keeps choosing
    tool_use will loop until your budget or your patience runs out.
    """
    messages: list[dict] = [{"role": "user", "content": goal}]

    for turn in range(1, max_turns + 1):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=TOOLS,
            messages=messages,
        )

        # Append the FULL content list, including tool_use blocks.
        messages.append({"role": "assistant", "content": response.content})

        tool_calls = [
            block for block in response.content if block.type == "tool_use"
        ]
        if verbose:
            names = [call.name for call in tool_calls]
            print(
                f"turn {turn}: stop_reason={response.stop_reason} "
                f"tools={names}"
            )

        if response.stop_reason != "tool_use":
            return extract_text(response)

        # One tool_result per tool_use, all in a single user message.
        results = []
        for call in tool_calls:
            output = TOOL_REGISTRY[call.name](**call.input)
            if verbose:
                print(f"         {call.name}({call.input}) -> {output}")
            results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": call.id,
                    "content": output,
                }
            )
        messages.append({"role": "user", "content": results})

    return f"STOPPED: hit the {max_turns}-turn limit before completing."


print(run_agent("What is the status of claim ABC-123?"))


### Parallel tool calls

A single response can contain several `tool_use` blocks. Handle them all
and return all results together — sending only the first will break the
conversation, because the API requires every `tool_use` to be answered.


In [ ]:
print(
    run_agent(
        "Compare claims ABC-123, XYZ-789 and DEF-456. Which paid most?"
    )
)


### Tool errors belong in `tool_result`, not in an exception

When a tool fails, do not let the exception escape the loop. Send the error
back as a `tool_result` with `is_error: True` and let the model adapt — ask
the user for a correction, try a different argument, or report the failure
cleanly. This is the "feed structured errors back to the model" pattern.


In [ ]:
def run_agent_with_error_handling(goal: str, max_turns: int = 6) -> str:
    """Tool-use loop that converts tool exceptions into tool_result errors."""
    messages: list[dict] = [{"role": "user", "content": goal}]

    for turn in range(1, max_turns + 1):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return extract_text(response)

        results = []
        for call in response.content:
            if call.type != "tool_use":
                continue
            try:
                output = TOOL_REGISTRY[call.name](**call.input)
                is_error = output.startswith("ERROR:")
            except (KeyError, TypeError) as error:
                output = f"ERROR: {type(error).__name__}: {error}"
                is_error = True

            print(f"turn {turn}: {call.name}({call.input}) -> {output}")
            results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": call.id,
                    "content": output,
                    "is_error": is_error,
                }
            )
        messages.append({"role": "user", "content": results})

    return f"STOPPED: hit the {max_turns}-turn limit."


print(run_agent_with_error_handling("Look up claim QQQ-999 for me."))


## 1.3 Agent Patterns — supervisor and subagents

A supervisor decomposes a task and dispatches subtasks to subagents. The
payoff is **context isolation**: each subagent gets a clean, narrow context
containing only what its subtask needs, instead of inheriting the whole
conversation.

Why that matters: a long shared context invites drift, costs more on every
turn, and lets irrelevant earlier content bias later answers. Isolation is
the architectural fix, and it is the reason the exam pairs "subagents" with
"context management" rather than with "speed".


In [ ]:
def plan_subtasks(task: str, count: int = 3) -> list[str]:
    """Supervisor step: decompose a task into independent subtasks.

    Structured outputs guarantee the shape. Note the schema root is an
    object, not an array -- so the list is wrapped in a `subtasks` key.
    """
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=f"Break the task into exactly {count} independent subtasks.",
        output_config={
            "format": {
                "type": "json_schema",
                "schema": {
                    "type": "object",
                    "properties": {
                        "subtasks": {
                            "type": "array",
                            "items": {"type": "string"},
                        }
                    },
                    "required": ["subtasks"],
                    "additionalProperties": False,
                },
            }
        },
        messages=[{"role": "user", "content": task}],
    )
    return json.loads(extract_text(response))["subtasks"]

## 1.4 Hooks: deterministic control over a non-deterministic system

A hook intercepts a tool call **before** it executes and can block it. This
is code, not persuasion — which is precisely why it works where a system
prompt does not. A prompt-level rule can be talked around by injected text;
an `if` statement cannot.

The pattern generalises to human-in-the-loop: block the call, surface it
for approval, execute only on a real approval.


In [ ]:
from dataclasses import dataclass


@dataclass
class HookDecision:
    """Result of a pre-tool-use check."""

    allowed: bool
    reason: str
    needs_human_approval: bool = False


AUTO_APPROVE_LIMIT = 1_000
BLOCKED_TOOLS = {"delete_claim"}


def pre_tool_use_hook(tool_name: str, tool_input: dict) -> HookDecision:
    """Decide whether a tool call may proceed.

    Three outcomes: allow, require human approval, or hard block. Deny by
    policy sits in code so no prompt can argue its way past it.
    """
    if tool_name in BLOCKED_TOOLS:
        return HookDecision(False, f"{tool_name} is disabled in this app")

    if tool_name == "issue_payment":
        amount = tool_input.get("amount", 0)
        if amount > AUTO_APPROVE_LIMIT:
            return HookDecision(
                False,
                f"${amount:,} exceeds the ${AUTO_APPROVE_LIMIT:,} "
                "auto-approval limit",
                needs_human_approval=True,
            )

    return HookDecision(True, "within policy")


def issue_payment(claim_id: str, amount: float) -> str:
    """The actual side-effecting operation."""
    return f"PAID ${amount:,.2f} on {claim_id}"


def guarded_call(tool_name: str, tool_input: dict) -> str:
    """Route every tool call through the hook before executing it."""
    decision = pre_tool_use_hook(tool_name, tool_input)

    if decision.allowed:
        return issue_payment(**tool_input)
    if decision.needs_human_approval:
        return f"HELD FOR APPROVAL: {decision.reason}"
    return f"BLOCKED: {decision.reason}"


print(guarded_call("issue_payment", {"claim_id": "ABC-123", "amount": 500}))
print(guarded_call("issue_payment", {"claim_id": "DEF-456", "amount": 15200}))
print(guarded_call("delete_claim", {"claim_id": "XYZ-789"}))


## 1.5 Managed vs. self-hosted agents

- **Self-hosted** — you run the loop, as in every cell above. Full control
  over execution, state, and infrastructure; you also own scaling, session
  persistence, retries, and observability.
- **Anthropic-hosted (Managed Agents)** — Anthropic runs the session.
  Billed on tokens plus session runtime. Less infrastructure work, less
  control over the execution environment.

The selection criteria are the usual build-versus-buy ones: how much you
need to customise the loop, whether your data may leave your environment,
and whether you want to own session state.
